# Query the Served Face Recognition Model

The retrained ONNX model is now deployed on **RHOAI** via **KServe RawDeployment** with **OpenVINO Model Server (OVMS)**.

In this notebook we query it through the **KServe v2 REST API** — the production inference pattern.
Instead of loading the model into local memory, we send images to the model server over HTTP
and receive bounding-box predictions back. This is how applications consume ML models in production.

In [ ]:
!pip install --no-cache-dir -q -r requirements.txt

import cv2
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
from IPython.display import Video, display
import remote_infer

## Configure the inference endpoint

KServe exposes each `InferenceService` as a Kubernetes Service with a predictable URL pattern:

```
http://<isvc-name>-predictor.<namespace>.svc.cluster.local:<port>/v2/models/<model-name>/infer
```

Update the URL below if your `InferenceService` name or namespace differs.

In [ ]:
# The KServe predictor service URL within the cluster.
# Format: http://<isvc-name>-predictor.<namespace>.svc.cluster.local:<port>/v2/models/<model-name>/infer
# CHANGE this if your InferenceService name or namespace differs.
INFER_URL = "http://face-recognition-predictor.private-ai.svc.cluster.local:8888/v2/models/face-recognition/infer"

# Quick readiness check
import requests
ready_url = INFER_URL.replace("/infer", "/ready")
try:
    resp = requests.get(ready_url, timeout=5)
    print(f"Model server status: {resp.json()}")
except Exception as e:
    print(f"Could not reach model server: {e}")
    print("Ensure the InferenceService is deployed and Ready:")
    print("  oc get inferenceservice face-recognition -n private-ai")

## Single image inference via REST API

In [ ]:
import glob

test_images = sorted(glob.glob("images/test_face_*.jpg")) + sorted(glob.glob("images/test_group_*.jpg"))
print(f"Testing {len(test_images)} images via REST API\n")

for image_path in test_images:
    annotated_img, detections = remote_infer.process_image(image_path, INFER_URL, conf_threshold=0.25)

    print(f"--- {image_path} ---")
    for det in detections:
        print(f"  Detected: {det['class_name']} (confidence: {det['confidence']:.2f})")

    img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    fig = plt.figure(figsize=(8, 5))
    plt.axis("off")
    plt.imshow(img_rgb)
    plt.title(f"{Path(image_path).name} — via Model Server")
    plt.show()

## Video inference via REST API

This demonstrates the **production inference pattern**: a client application extracts frames from a video stream,
sends each frame to the model server over HTTP, and reassembles the annotated results.

This is slower than local inference (network round-trip per frame) but scales independently —
the model server can be scaled horizontally to handle many concurrent clients.

In [ ]:
video_path = "videos/test_group_video.mov"

if not Path(video_path).exists():
    print(f"Video not found: {video_path}")
    print("Place a test video in the videos/ folder.")
else:
    print("Processing video via REST API — this may take several minutes...")
    output_path = remote_infer.process_video_rest(video_path, INFER_URL, conf_threshold=0.25)
    display(Video(output_path, embed=True, width=640))

## Conclusion

Over the course of this notebook series we have:

1. **Explored** YOLO11 face detection — understanding how the model detects faces in images
2. **Learned** to draw bounding boxes and inspect detection metadata
3. **Retrained** YOLO11 on custom face data with auto-annotation to recognize a specific person
4. **Tested** the retrained model locally on still images and video
5. **Deployed** to RHOAI via KServe + OpenVINO and queried the model via REST API

This demonstrates **RHOAI 3.3's** ability to handle both **generative AI** (LLMs via vLLM) and
**predictive AI** (computer vision via OpenVINO) workloads on the same platform.